# 1) Download and save SATCAT


In [1]:
# Inputs: none | Process: import libs | Outputs: env ready
import requests
import pandas as pd
from io import StringIO
import time
from pathlib import Path
from datetime import datetime


In [ ]:
# Inputs: env vars or hardcoded creds | Process: set config | Outputs: constants
USERNAME = "aaeushsingh98@gmail.com"
PASSWORD = "RA5wtMpC67!!6r6AB12"
BATCH_SIZE = 100000
THROTTLE_SECONDS = 1
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
for d in [RAW_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)


In [ ]:
# Inputs: creds + batch config | Process: login, find max NORAD, fetch by ranges | Outputs: satcat_df
with requests.Session() as session:
    login_url = "https://www.space-track.org/ajaxauth/login"
    payload = {"identity": USERNAME, "password": PASSWORD}
    r = session.post(login_url, data=payload)
    if r.status_code != 200 or "Failed" in r.text:
        raise RuntimeError("Login failed")

    max_id_url = (
        "https://www.space-track.org/basicspacedata/query/class/satcat/"
        "orderby/NORAD_CAT_ID desc/limit/1/format/csv"
    )
    resp = session.get(max_id_url)
    resp.raise_for_status()
    max_norad_id = int(pd.read_csv(StringIO(resp.text))["NORAD_CAT_ID"].iloc[0])

    all_data = []
    for start_id in range(1, max_norad_id + 1, BATCH_SIZE):
        end_id = min(start_id + BATCH_SIZE - 1, max_norad_id)
        url = (
            f"https://www.space-track.org/basicspacedata/query/class/satcat/"
            f"NORAD_CAT_ID/{start_id}--{end_id}/orderby/NORAD_CAT_ID asc/format/csv"
        )
        resp = session.get(url)
        resp.raise_for_status()
        all_data.append(pd.read_csv(StringIO(resp.text)))
        time.sleep(THROTTLE_SECONDS)

satcat_df = pd.concat(all_data, ignore_index=True)
satcat_df['NORAD_CAT_ID'] = pd.to_numeric(satcat_df['NORAD_CAT_ID'], errors='coerce')
satcat_df = satcat_df.dropna(subset=['NORAD_CAT_ID']).drop_duplicates(subset=['NORAD_CAT_ID']).reset_index(drop=True)


In [ ]:
# Inputs: satcat_df | Process: save raw+latest | Outputs: CSV paths
TS = datetime.now().strftime('%Y%m%d_%H%M%S')
raw_path = RAW_DIR / f"satcat_{TS}.csv"
latest_path = PROCESSED_DIR / "satcat_latest.csv"
satcat_df.to_csv(raw_path, index=False)
satcat_df.to_csv(latest_path, index=False)
print("Saved:", raw_path, "and", latest_path)
